# Moving Averages, Seasonal, and Naive Forecasting Approaches: Baselines and Evaluation

Your manager wants a benchmark before approving a more complicated model. We will compare three simple methods using rolling one-month-ahead forecasts over the same future year.

For each test month, pretend that the actual sales from all earlier months are already known. This matches the usual Excel approach: use the previous month for the naive forecast and the previous three months for the moving-average forecast.

**How to work:** Run one cell at a time. Read the explanation, inspect the output, and answer the interpretation questions in your own words.

## Task 1 - Load and split chronologically

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')


from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

data = pd.read_csv('monthly_retail_sales_clean.csv', parse_dates=['date']).sort_values('date').reset_index(drop=True)
train = data.iloc[:-12].copy() #copy all except last year
test = data.iloc[-12:].copy() #copy last year only for testing
print(train['date'].min(), 'to', train['date'].max())
print(test['date'].min(), 'to', test['date'].max())

**Write:** Why are the last 12 months used as testing rather than 12 random rows?

## Task 2 - Build three rolling one-step-ahead baselines

The test set is still kept separate for fair evaluation. However, after a test month has occurred, its actual value becomes available and may be used to forecast the following month.

The important rule is: **never use the current month's actual sales to predict that same month.** We use `shift()` to move past values forward by one row before creating each forecast.

In [ ]:
# Build forecasts from the complete chronological series.
# We will select only the test-year rows after creating the lagged values.
sales = data['sales']

# Naive: forecast each month using the immediately previous month's actual sales.
naive = sales.shift(1).loc[test.index].to_numpy()

# Seasonal naive: forecast each month using the same month one year earlier.
seasonal_naive = sales.shift(12).loc[test.index].to_numpy()

# 3-month moving average: shift first, then average the previous 3 actual months.
# January uses Oct-Nov-Dec; February uses Nov-Dec-Jan; and so on.
moving_avg = sales.shift(1).rolling(window=3).mean().loc[test.index].to_numpy()

# Show exactly which earlier values produced each forecast.
forecast_table = test[['date', 'sales']].copy()
forecast_table['Previous month'] = naive
forecast_table['Naive forecast'] = naive
forecast_table['Previous 3-month values'] = [
    ', '.join(f'{value:,.0f}' for value in sales.iloc[i-3:i])
    for i in test.index
]
forecast_table['3-month moving-average forecast'] = moving_avg
forecast_table['Seasonal-naive forecast'] = seasonal_naive
forecast_table.rename(columns={'sales': 'Actual sales'}).round(2)

## Task 3 - Create one evaluation function

In [ ]:
def evaluate(actual, forecast):
    return {
        'MAE': mean_absolute_error(actual, forecast),
        'RMSE': np.sqrt(mean_squared_error(actual, forecast)),
        'MAPE_pct': mean_absolute_percentage_error(actual, forecast) * 100
    }

results = pd.DataFrame({
    'Naive': evaluate(test['sales'], naive),
    'Seasonal naive': evaluate(test['sales'], seasonal_naive),
    '3-month average': evaluate(test['sales'], moving_avg)
}).T.round(2)
results

**Write:** Which baseline has the smallest MAE? Explain that MAE in Canadian dollars.

## Task 4 - Compare forecasts visually

In [ ]:
#plotting original data
data = data.sort_values('date')
plt.figure(figsize=(12, 4))
plt.plot(data['date'], data['sales'], color='teal')
plt.title('Coastal Outfitters Monthly Sales')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(train['date'].tail(24), train['sales'].tail(24), label='Training history', color='gray')
plt.plot(test['date'], test['sales'], label='Actual test sales', color='black', linewidth=2.5)
plt.plot(test['date'], naive, label='Naive')
plt.plot(test['date'], seasonal_naive, label='Seasonal naive')
plt.plot(test['date'], moving_avg, label='3-month average')
plt.legend()
plt.title('Baseline Forecast Comparison')
plt.show()

**Write:** Which method follows the seasonal peaks most closely? 

## Task 5 - Inspect residuals for the best baseline

In [ ]:
forecasts = {
    'Naive': naive,
    'Seasonal naive': seasonal_naive,
    '3-month average': moving_avg
}
best_name = results['MAE'].idxmin()
best_forecast = forecasts[best_name]
residuals = test['sales'].to_numpy() - best_forecast
plt.figure(figsize=(10, 3))
plt.axhline(0, color='black', linewidth=1)
plt.plot(test['date'], residuals, marker='o')
plt.title(f'{best_name} Residuals: Actual - Forecast')
plt.show()
print('Best baseline by MAE:', best_name)
print('Average residual:', residuals.mean().round(2))

## Exit ticket
Recommend one baseline. Include the metric, its business meaning, one visual observation, and one limitation.